## Additional Exploration

This Jupyter Notebook takes a closer look at the indicators Landcover/Landuse that come with LUCAS data, the Corine Landcover data extracted for our set of points as well as Soil Type information that was extracted from BGR (Soil Regions of the European Union and Adjacent Countries 1 : 5 000 000). The script creates a frequency table for the Landcover and Landuse or Corine_Landcover columns, showing the number of occurrences of each category and the total number of distinct categories. The amount of possible categories (i.e. forest, agriculture, water or fallow) per indicator (i.e. landcover) needs to be reduced so that ML later has only few categories to deal with (3 to 5). Certain untypical or meaningless categories (and points respectively) can be excluded from the dataset. Final aim of this Notebook is to offer information to identify these points. A similar process and exploration is done for soil type information (), the data in Indep_indicators.csv shows a huge diversity of soil types. This might even lead to the exclusion of this indicator.


### Content
1. Merging Points & SHI with independent indicators (landcover, landuse, ...)
2. Counting occurences per indicator
3. Pivot or Cross Tables

In [ ]:
import pandas as pd
import csv
import collections
import openpyxl
#import numpy as np

### 1. Merging Points & SHI with independent indicators (landcover, landuse, ...)

In [ ]:
# load data
df_shi_lc_lu = pd.read_csv("../output/df_transform_final_SHI.csv")
df_indep_indicators = pd.read_csv("01_input/Indep_indicators.csv")

# drop some columns not needed now
df_shi_lc_lu.drop(columns=['Clay_SOC_score', 'OC_score', 'P_score', 'N_score',
       'K_score', 'bulk_density_score', 'EC_score', 'erosion_score',
       'pH_score'], inplace=True)

# merge
df_merged = df_shi_lc_lu.merge(
    df_indep_indicators,
    on='POINT_ID',
    how='left',
    suffixes=('', '_ii')  # avoids clashes if columns match
)

# replace all missing values with NaN
df_merged = (
    df_merged
    .replace(r'^\s*$', pd.NA, regex=True)
    .fillna('NO DATA')
)

# reorder columns
df_merged = df_merged.loc[:, ['POINT_ID', 'SHI', 'hoehe_m', 'Landuse', 'Landcover', 'Corine_Landcover', 'DO_RSG1', 'DO_MAT1']]

# save & check
df_merged.to_csv("../output/df_points_indep_indicators.csv", index=False)
df_merged.head(15)

### 2. Counting occurences per indicator

In [ ]:
def count_occurences_per_indicator(col):
    counts = collections.Counter()

    with open('../output/df_points_indep_indicators.csv', 'r', newline='') as input_file:
        reader = csv.DictReader(input_file)

        for row in reader:
            value = row[col].strip()
            counts[value] += 1

    return counts


indicators_columns = ['Landuse', 'Landcover', 'Corine_Landcover', 'DO_RSG1', 'DO_MAT1']

for column in indicators_columns:
    counts = count_occurences_per_indicator(column)

    print(f"\n=== {column} ({len(counts)} categories) ===")

    for category, count in counts.most_common():
        print(f"  {category:<58} {count}")

### 3. Pivot or Cross Tables

Crosstables can be created and printed or exportet as csv or xlsx here. From experience it is a better approach to import a csv directly into Excel and create a pivot table there. Functionality is better, columns can be hidden or shown and counted values will adapt. To do so just import df_points_indep_indicators.csv in Excel and use pivot table function there.

In [ ]:
# Example of creating crosstable, use the two columns you are interested in cross counting
# some possible columns: 'Landuse', 'Landcover', 'Corine_Landcover', 'DO_RSG1', 'DO_MAT1'
df = pd.read_csv('../output/df_points_indep_indicators.csv')

crosstab = pd.crosstab(df['Landcover'], df['Corine_Landcover'], margins=True)
print(crosstab)

# this file can be used to create pivot tables in Excel
# df.to_excel("../output/df_points_indep_indicators.xlsx", index=False)